#### > Autograd(자동미분)
: PyTorch 에서 연산과정을 추적하고 역전파를 통해 미분값을 자동으로 계산하는 기능
 - **미분 계산 자동화**
 - **기울기 계산** : 각 입력값과 모델의 가중치가 손실함수에 미치는 영향을 계산
 - **모델 학습 지원** : 계산한 기울기를 optimizer 에 전달하여 가중치를 업데이트

In [1]:
import torch
import pandas as pd
import numpy as np
from torch.utils.data import TensorDataset, DataLoader

##### > Autograd 의 기본 흐름

1. 미분할 텐서에 requires_grad = True 설정
2. 해당 텐서를 사용하여 연산 수행
3. 결과 텐서에 backward() 실행. -> 거꾸로 올라가면서 미분
4. 원본 텐서의 grad 속성에서 미분값 확인 가능

#### 1. requires_grad
- True : 해당 텐서에 대한 연산을 추적하여 기울기를 계산할 수 있도록 설정.
- False : 기본값, 연산을 추적하지 않음
- 일반적으로 모델이 학습해야 하는 가중치(weight)와 편향(bias)에 True를 설정함

In [2]:
x = torch.tensor(1.0, requires_grad= True)
print(x)

tensor(1., requires_grad=True)


In [ ]:
x = torch.tensor(1.0)   # requires_grad= False
print(x)
print(x.requires_grad)

x.requires_grad=True    # 수정 가능
print(x.requires_grad)

x.requires_grad=False   # 재수정(turn off) 가능
print(x.requires_grad)

tensor(1.)
False
True
False


In [ ]:
# 1. 텐서 생성
x = torch.tensor([1.0], requires_grad=True)

# 2. 함수 정의
y = x**2
print(y) # tensor([1.], grad_fn=<PowBackward0>) -> 제곱을 했음을 추적

# 3. 역전파 실행
y.backward()

# 4. 결과와 미분값(기울기, grad)
print('y: ', y)                # y:  tensor([1.], grad_fn=<PowBackward0>)
print('x의 기울기: ', x.grad)   # x의 기울기:  tensor([2.])

tensor([1.], grad_fn=<PowBackward0>)
y:  tensor([1.], grad_fn=<PowBackward0>)
x의 기울기:  tensor([2.])


In [ ]:
x = torch.tensor(1.0, requires_grad=True)
x.requires_grad=False
    # transfer learning 할 때 사용
    # 내가 만들어 놓은 학습된 모델에서 일부분만 학습시키고 싶을 때 나머지 부분의 추적을 off
y = x**2
print(y)
# y.backward() error

tensor(1.)


In [ ]:
x = torch.tensor([2., 4], requires_grad=True)
y = x * 3
print('x.requires_grad:', x.requires_grad)
print('y.requires_grad:', y.requires_grad)
print('x.is_leaf:', x.is_leaf) # leaf 여부 
print('y.is_leaf:', y.is_leaf)
print('y.grad_fn:', y.grad_fn) # 어떤 연산을 통해 y 를 도출했는지 설명 (MulBackward0 : 곱셈 연산을 했음)

x.requires_grad: True
y.requires_grad: True
x.is_leaf: True
y.is_leaf: False
y.grad_fn: <MulBackward0 object at 0x0000022BB0D28C10>


### Common grad_fn results

| operation | result |
| --- | --- |
|Addition: |AddBackward0|
|Multiplication: |MulBackward0|
|Power: |PowBackward0|
|ReLU: |ReluBackward0|
|Mean: |MeanBackward0|
|Unsqueeze: |UnsqueezeBackward0|
|nn.Linear: |AddmmBackward0|
|Sigmoid: |SigmoidBackward0|

#### 2. 연쇄법칙(Chain Rule)
: 함수가 여러 연산으로 연결되어 있으면 각 연산의 미분값을 연쇄적으로 곱하여 최종 기울기를 계산함

In [ ]:
x = torch.tensor(1., requires_grad=True)
y = x**2
print(y)

z = 3 * y
print(z)

z.backward()
print(x.grad) # 최종 미분값 할당은 leaf tensor 인 x 에만
print(y.grad) # None

tensor(1., grad_fn=<PowBackward0>)
tensor(3., grad_fn=<MulBackward0>)
tensor(6.)
None


C:\Users\mega\AppData\Local\Temp\ipykernel_9836\4062033234.py:10: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\build\aten\src\ATen/core/TensorBody.h:493.)
  print(y.grad)


In [15]:
# 중간에 있는 y의 미분값도 알고 싶다면
x = torch.tensor(1., requires_grad=True)
y = x ** 2
y.retain_grad() # y의 grad도 축적

z = 3 * y

z.backward()
print(x.grad)
print(y.grad)

tensor(6.)
tensor(3.)


In [16]:
x = torch.tensor(1., requires_grad=True)
y = x ** 2
z = 3 * y

y.backward()  # z 연산을 수행했지만, y에서 역전파하는 경우
print(x.grad)

tensor(2.)


In [ ]:
x = torch.tensor(1., requires_grad=True)
a = x ** 2
b = a + 1
print(b) # AddBackward0

c = b**2
c.backward()
print(x.grad) # x^4 + 2x^2 + 1 의 미분후 값 4*1^3 + 4*1^1 = 8

tensor(2., grad_fn=<AddBackward0>)
tensor(8.)


#### 3. 편미분
: 함수에 여러 변수가 있을 때는 각 변수에 대해 따로 미분할 수 있음.

In [ ]:
x = torch.tensor(1., requires_grad=True)
# y = torch.tensor(1., requires_grad=True)
y = torch.tensor(2., requires_grad=True)
z = 2 * x ** 2 + y ** 2
z.backward()
print(x.grad) # x에대해 편미분한 값
print(y.grad) # y에대해 편미분한 값

tensor(4.)
tensor(4.)


In [22]:
x = torch.tensor(1., requires_grad=True)
y = torch.tensor(1., requires_grad=True)
z = x ** 2 + 2 * x * y + y ** 2
z.backward()
print(x.grad)
print(y.grad)

tensor(4.)
tensor(4.)


#### 4. 벡터 출력의 기울기 계산
- 방법 1: <br>
y.sum().backward() <br><br>
- 방법 2: <br>
y.backward(torch.ones_like(y))

In [28]:
x = torch.tensor([1., 2, 3], requires_grad=True)
y = torch.sum(x ** 2)
y.backward()
print(y)
print(x.grad)

x = torch.tensor([1., 2, 3], requires_grad=True)
y = x ** 2
y.sum().backward()
print(y)
print(x.grad)

x = torch.tensor([1., 2, 3], requires_grad=True)
y = x ** 2
y.backward(torch.ones_like(y))
print(y)
print(x.grad)

tensor(14., grad_fn=<SumBackward0>)
tensor([2., 4., 6.])
tensor([1., 4., 9.], grad_fn=<PowBackward0>)
tensor([2., 4., 6.])
tensor([1., 4., 9.], grad_fn=<PowBackward0>)
tensor([2., 4., 6.])


#### 5. gradient 누적과 초기화
: backward()를 여러번 실행하면 grad에 새로운 기울기가 더해짐
- 기울기를 새로 계산하려면 기존 기울기를 0으로 초기화 필요
    - x.grad.zero_()
    - 모델 학습에서는 일반적으로 optimizer.zero_grad()를 사용함

In [33]:
# backward() 누적
x = torch.tensor([1., 2, 3], requires_grad=True)
y = (x ** 2).sum()

y.backward()
print('첫 번째 역전파: ', x.grad)

z = (3 * x).sum()
z.backward()
print('두 번째 역전파: ', x.grad)

# grad 초기화
x.grad.zero_()
print('초기화 후:', x.grad)

# 초기화 후 새로 계산
w = (x ** 3).sum()
w.backward()
print('초기화 후 새로 계산:', x.grad)

첫 번째 역전파:  tensor([2., 4., 6.])
두 번째 역전파:  tensor([5., 7., 9.])
초기화 후: tensor([0., 0., 0.])
초기화 후 새로 계산: tensor([ 3., 12., 27.])


#### 6. torch.autograd.grad()
: 지정한 입력 텐서에 대한 기울기(미분값)을 반환하는 함수

- backward(): 계산한 기울기를 입력 텐서의 .grad에 누적
- torch.autograd.grad(): 기울기를 반환해줌. .grad에 누적하지 않음

- usage
1. 특정 변수의 미분값만 필요할 때
2. gradient를 분리하여 저장(기울기를 .grad 에 저장하고 싶지 않음)
3. 여러 변수의 편미분을 동시에 구할 때: 결과값을 튜플로 반환

- format
1. torch.autograd.grad(outputs, inputs)

In [ ]:
x = torch.tensor(4., requires_grad=True)
y = x ** 3

gradient = torch.autograd.grad(y, x)
print('반환된 기울기:', gradient)
print('x.grad:', x.grad)    #backward()하지 않았음

반환된 기울기: (tensor(48.),)
x.grad: None


#### 7. detach() : Returns a tensor detached from the computation graph
requires_grad= True 인 텐서에 연산을 적용하면 PyTorch는 연산과정을 추적함 <br>
그런데 특정 텐서부터는 더 이상 gradient를 계산하지 않도록 하고 싶을 때 detach()사용 <br>
**기존 텐서와 값은 공유하지만, 자동 미분 계산 그래프에서 분리된 텐서를 반환**

- 원본 텐서의 연산 추적은 그대로 유지됨
- 분리된 텐서는 해당 연산 그래프를 통한 기울기 계산에 연결되지 않음
- 학습 결과를 출력하거나 NumPy로 변환할 때 자주 사용.

In [39]:
x = torch.tensor([1., 2, 3], requires_grad=True)
y = x * 2
y_detach = y.detach()

print('y:', y)
print('y.requires_grad:', y.requires_grad)
print('y_detach:', y_detach)
print('y_detach.requires_grad:', y_detach.requires_grad)
print('y_detach.grad_fn:', y_detach.grad_fn)

y: tensor([2., 4., 6.], grad_fn=<MulBackward0>)
y.requires_grad: True
y_detach: tensor([2., 4., 6.])
y_detach.requires_grad: False
y_detach.grad_fn: None


In [ ]:
# 변수의 값을 그냥 상수로 사용하고 싶을 때
x = torch.tensor(1., requires_grad=True)
y = x**2
# z = y 로 넣으면 수식이 복사되어 들어가기 때문에 상수로 사용 불가
z = y.detach()  # z는 이제 상수의 주소를 출력

w = y + z   # x ** 2 + 1 | x ** 2 + x ** 2 가 아님
w.backward()
print(x.grad)   # tensor(2.) | tensor(4.) 이 아님

tensor(2.)


#### 8. torch.no_grad()
: 지정한 블록 코드 안에서는 자동 미분을 위한 연산 추적을 비활성화
- 검증이나 예측 등(학습이 아닌) 기울기 계산이 필요하지 않은 상황
- 학습 중 모델의 가중치를 업데이트 할 때
- 불필요한 계산 그래프 생성을 막아 메모리 사용량을 줄임

[사용형식]
```python
with torch.no_grad() :
    코드 블록
    # 기울기를 추적하지 않는 연산
```

- `detach()`는 특정 텐서를 계산 그래프에서 분리
- `no_grad()`는 특정 코드 블록의 연산 추적을 막음

In [44]:
x = torch.tensor(3., requires_grad=True)
y1 = x ** 2
with torch.no_grad():
    y2 = x ** 2
    z = 2* y2

print('일반 연산 requires_grad:', y1.requires_grad)
print('no_grad 내부 연산의 requires_grad:', y2.requires_grad)
print('y2.grad_fn:', y2.grad_fn)

print('no_grad 내부 연산의 requires_grad:', z.requires_grad)
print('z.grad_fn:', z.grad_fn)

일반 연산 requires_grad: True
no_grad 내부 연산의 requires_grad: False
y2.grad_fn: None
no_grad 내부 연산의 requires_grad: False
z.grad_fn: None


- `torch.no_grad()`
    - 특정 코드블록에 적용
    - 블록 내부의 연산 그래프 추적을 중단
    - 블록이 끝나면 기존상태로 복귀
    - torch.enable_grad(). 코드 블럭 내부에서 다시 추적 요청 가능
- `requires_grad=False`
    - 특정 텐서에 적용
    - 해당 텐서에 대한 기울기 추적을 비활성화
    - 다시 True로 변경하기 전까지 유지

In [45]:
x = torch.tensor(1., requires_grad=True)

with torch.no_grad():
    y = x**2
    print(x.requires_grad)
    print(y)

print('x 추적여부:',x.requires_grad)

y = x ** 2
print(y)
print('y 추적여부:',y.requires_grad)

print('-'*30)

x = torch.tensor(1., requires_grad=True)
x.requires_grad= False
y = x**2
print('x 추적여부:',x.requires_grad)
print(y)

True
tensor(1.)
x 추적여부: True
tensor(1., grad_fn=<PowBackward0>)
y 추적여부: True
------------------------------
x 추적여부: False
tensor(1.)


In [54]:
# grad 여부 그림으로 표시
%pip install torchviz graphviz

Note: you may need to restart the kernel to use updated packages.


In [56]:
from torchviz import make_dot
from graphviz import graphs

In [57]:
x = torch.tensor(1., requires_grad=True)
make_dot(x)

ExecutableNotFound: failed to execute WindowsPath('dot'), make sure the Graphviz executables are on your systems' PATH